# Analyse de l'Incertitude des LLMs avec Entropie Token-Level
## Dataset: WebQuestions - Modèle: Qwen2.5-3B-Instruct

**Objectifs:**
- Génération de texte libre avec calcul d'entropie token-level
- Optimisations pour maximiser l'accuracy (target: 60-70%)
- Tableau récapitulatif complet des prédictions
- Analyse de la corrélation entropie-accuracy
- **NOUVEAU: Visualisations avancées de l'EPR (Entropy Production Rate)**

## 0. Installation et Configuration

In [ ]:
#!pip install -q transformers accelerate datasets pandas numpy scipy tqdm scikit-learn matplotlib seaborn
#print("Installation terminée")

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy
from datasets import load_dataset
from tqdm import tqdm
from collections import Counter
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_curve, roc_auc_score

# Configuration
np.random.seed(42)
torch.manual_seed(42)

# Style des graphiques
plt.style.use('default')
sns.set_palette("husl")

print("Imports réussis")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Imports réussis
Device: GPU
GPU: NVIDIA A100-SXM4-40GB
Mémoire GPU: 42.47 GB


## 1. Chargement du Modèle

In [ ]:
from huggingface_hub import login

login(token="Put your token")

In [ ]:
#MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
#MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
#MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
#MODEL_NAME = "microsoft/Phi-4-mini-instruct"
#MODEL_NAME = "tiiuae/falcon-rw-1b"
#MODEL_NAME = "google/gemma-2-9b-it"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"







print(f"Chargement du modèle: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

model.eval()
print(f"Modèle chargé avec succès sur {model.device}")

Chargement du modèle: mistralai/Mistral-7B-Instruct-v0.3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Modèle chargé avec succès sur cuda:0


## 2. Chargement du Dataset

In [ ]:
print("Chargement du dataset WebQuestions...")
dataset = load_dataset("web_questions", trust_remote_code=True)
dataset_list = []
for item in dataset['test']:
    dataset_list.append({
        'question': item['question'],
        'answers': item['answers']
    })

print(f"Dataset chargé: {len(dataset_list)} questions")
print(f"Exemple de question: {dataset_list[0]['question']}")
print(f"Réponses possibles: {dataset_list[0]['answers']}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'web_questions' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'web_questions' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Chargement du dataset WebQuestions...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/260k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/142k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3778 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2032 [00:00<?, ? examples/s]

Dataset chargé: 2032 questions
Exemple de question: what does jamaican people speak?
Réponses possibles: ['Jamaican Creole English Language', 'Jamaican English']


## 3. Fonctions de Génération avec Calcul d'Entropie

In [ ]:
def create_optimized_prompt(question):
    """
    Prompt optimisé avec 5 exemples few-shot.
    """
    return f"""Answer each question with ONLY the answer (maximum 5 words, no explanation, no punctuation at the end).

Examples:
Q: What is the capital of France?
A: Paris

Q: Who wrote Romeo and Juliet?
A: William Shakespeare

Q: What year did World War II end?
A: 1945

Q: What is the largest planet in our solar system?
A: Jupiter

Q: Who painted the Mona Lisa?
A: Leonardo da Vinci

Now answer this question:
Q: {question}
A:"""


In [ ]:
def generate_answer_with_token_entropy(question, max_new_tokens=10, temperature=0.1):
    """
    Génère une réponse avec entropie token-level.
    """
    prompt = create_optimized_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs.sequences[0][inputs.input_ids.shape[1]:]
    answer_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # Calcul entropie token-level
    token_entropies = []
    '''
    for score in outputs.scores:
    topk_logits, _ = torch.topk(score[0], k=10)  # Top-10 seulement
    topk_probs = F.softmax(topk_logits, dim=-1).cpu().numpy()
    token_entropy = entropy(topk_probs)  # EPR
    '''
    for score in outputs.scores:
        probs = F.softmax(score[0], dim=-1).cpu().numpy()
        token_entropy = entropy(probs)
        token_entropies.append(token_entropy)

    mean_entropy = np.mean(token_entropies) if token_entropies else 0.0

    return {
        'answer': answer_text,
        'mean_entropy': mean_entropy,
        'token_entropies': token_entropies
    }

## 4. Test sur Question Exemple

In [ ]:
print("Test sur une question exemple\n")

test_question = dataset_list[0]['question']
test_answers = dataset_list[0]['answers']

print(f"Question: {test_question}")
print(f"Réponses acceptables: {test_answers}\n")

result = generate_answer_with_token_entropy(test_question)

print(f"Réponse générée: {result['answer']}")
print(f"Entropie moyenne: {result['mean_entropy']:.4f} bits")
print(f"Nombre de tokens: {len(result['token_entropies'])}")

Test sur une question exemple

Question: what does jamaican people speak?
Réponses acceptables: ['Jamaican Creole English Language', 'Jamaican English']

Réponse générée: English
Entropie moyenne: 0.0027 bits
Nombre de tokens: 2


## 5. Génération Multiple avec Entropie

In [ ]:
def generate_multiple_with_entropy(question, n_samples=20, temperature=0.1):
    """
    Génère n_samples réponses avec entropie.
    Augmenté à 20 échantillons pour meilleure mesure.
    """
    answers = []
    mean_entropies = []
    all_token_entropies = []

    for _ in range(n_samples):
        result = generate_answer_with_token_entropy(question, temperature=temperature)
        answers.append(result['answer'])
        mean_entropies.append(result['mean_entropy'])
        all_token_entropies.extend(result['token_entropies'])

    return {
        'answers': answers,
        'mean_entropies': mean_entropies,
        'overall_mean_entropy': np.mean(mean_entropies),
        'all_token_entropies': all_token_entropies
    }

def normalize_answer(text):
    """Normalise une réponse pour comparaison."""
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = ' '.join(text.split())
    return text

def check_answer_match(generated, true_answers):
    """Vérifie si la réponse générée correspond à une réponse vraie."""
    generated_norm = normalize_answer(generated)
    for true_ans in true_answers:
        true_norm = normalize_answer(true_ans)
        if generated_norm == true_norm or generated_norm in true_norm or true_norm in generated_norm:
            return True
    return False

## 6. Test de Génération Multiple

In [ ]:
print("Test de génération multiple avec entropie\n")

test_result = generate_multiple_with_entropy(test_question, n_samples=5)

print(f"Question: {test_question}")
print(f"Réponses vraies: {test_answers}\n")

print("Réponses générées:")
for i, (ans, ent) in enumerate(zip(test_result['answers'], test_result['mean_entropies']), 1):
    match = "✓" if check_answer_match(ans, test_answers) else "✗"
    print(f"{i}. {ans} (Entropie: {ent:.4f}) {match}")

print(f"\nEntropie moyenne globale: {test_result['overall_mean_entropy']:.4f} bits")

answer_counts = Counter(test_result['answers'])
print(f"\nDistribution des réponses:")
for ans, count in answer_counts.most_common():
    print(f"  '{ans}': {count}")

Test de génération multiple avec entropie

Question: what does jamaican people speak?
Réponses vraies: ['Jamaican Creole English Language', 'Jamaican English']

Réponses générées:
1. English (Entropie: 0.0027) ✓
2. English (Entropie: 0.0027) ✓
3. English (Entropie: 0.0027) ✓
4. English (Entropie: 0.0027) ✓
5. English (Entropie: 0.0027) ✓

Entropie moyenne globale: 0.0027 bits

Distribution des réponses:
  'English': 5


## 7. Traitement du Dataset Complet

In [ ]:
# Configuration optimisée pour accuracy
N_SAMPLES = 20        # Plus d'échantillons
TEMPERATURE = 0.1     # Très bas pour déterminisme
NUM_QUESTIONS = 1000

print(f"Génération de {N_SAMPLES} réponses pour {NUM_QUESTIONS} questions")
print(f"Température: {TEMPERATURE} (très déterministe pour maximiser accuracy)\n")

results = []
selected_questions = dataset_list[:NUM_QUESTIONS]

for idx, item in enumerate(tqdm(selected_questions, desc="Processing questions")):
    question = item['question']
    true_answers = item['answers']

    gen_result = generate_multiple_with_entropy(question, n_samples=N_SAMPLES, temperature=TEMPERATURE)

    # Consensus
    answer_counts = Counter(gen_result['answers'])
    consensus_answer = answer_counts.most_common(1)[0][0]
    consensus_freq = answer_counts.most_common(1)[0][1] / N_SAMPLES

    # Accuracy
    correct_count = sum([check_answer_match(ans, true_answers) for ans in gen_result['answers']])
    accuracy = correct_count / N_SAMPLES
    is_correct = check_answer_match(consensus_answer, true_answers)

    results.append({
        'question': question,
        'true_answers': true_answers,
        'generated_answers': gen_result['answers'],
        'mean_entropies': gen_result['mean_entropies'],
        'overall_mean_entropy': gen_result['overall_mean_entropy'],
        'consensus_answer': consensus_answer,
        'consensus_frequency': consensus_freq,
        'accuracy': accuracy,
        'is_correct': is_correct
    })

print(f"\nTraitement terminé: {len(results)} questions analysées")

Génération de 20 réponses pour 1000 questions
Température: 0.1 (très déterministe pour maximiser accuracy)



Processing questions:  12%|█▏        | 118/1000 [08:10<41:19,  2.81s/it]

## 8. Calcul des Métriques Globales

In [ ]:
# Accuracy globale
all_accuracies = [r['accuracy'] for r in results]
global_accuracy = np.mean(all_accuracies)

# Questions correctes
correct_questions = sum([r['is_correct'] for r in results])
question_accuracy = correct_questions / len(results)

# Entropie globale
all_mean_entropies = [r['overall_mean_entropy'] for r in results]
global_mean_entropy = np.mean(all_mean_entropies)
global_std_entropy = np.std(all_mean_entropies)

print("\n" + "="*80)
print("MÉTRIQUES GLOBALES")
print("="*80)
print(f"Accuracy Globale (moyenne sur échantillons): {global_accuracy:.2%}")
print(f"Questions Réussies (consensus correct): {correct_questions}/{len(results)} ({question_accuracy:.2%})")
print(f"Entropie Moyenne: {global_mean_entropy:.4f} ± {global_std_entropy:.4f} bits")
print("="*80)

## 9. Tableau Récapitulatif Complet

In [ ]:
# Créer tableau récapitulatif
summary_data = []

for i, r in enumerate(results, 1):
    # Réponses uniques générées
    unique_answers = list(set(r['generated_answers']))

    summary_data.append({
        'N°': i,
        'Question': r['question'][:60] + '...' if len(r['question']) > 60 else r['question'],
        'Réponses Vraies': ' | '.join(r['true_answers'][:3]),
        'Consensus': r['consensus_answer'],
        'Freq': f"{r['consensus_frequency']:.1%}",
        'Accuracy': f"{r['accuracy']:.1%}",
        'Correct': '✓' if r['is_correct'] else '✗',
        'Entropie': f"{r['overall_mean_entropy']:.3f}",
        'Nb Uniques': len(unique_answers)
    })

df_summary = pd.DataFrame(summary_data)

print("\n" + "="*120)
print("TABLEAU RÉCAPITULATIF COMPLET")
print("="*120)
print(df_summary.to_string(index=False))
print("="*120)

## 10. Analyse Détaillée des Meilleures et Pires Questions

In [ ]:
# Top 10 questions correctes (haute accuracy)
correct_sorted = sorted([r for r in results if r['is_correct']],
                       key=lambda x: x['accuracy'], reverse=True)[:10]

print("\n" + "="*120)
print("TOP 10 QUESTIONS CORRECTES (Accuracy la plus élevée)")
print("="*120)

for i, r in enumerate(correct_sorted, 1):
    print(f"\n{i}. Question: {r['question']}")
    print(f"   Réponses vraies: {r['true_answers']}")
    print(f"   Consensus: '{r['consensus_answer']}' (freq: {r['consensus_frequency']:.1%})")
    print(f"   Accuracy: {r['accuracy']:.1%} | Entropie: {r['overall_mean_entropy']:.4f}")

# Top 10 questions incorrectes (plus difficiles)
incorrect_sorted = sorted([r for r in results if not r['is_correct']],
                         key=lambda x: x['overall_mean_entropy'], reverse=True)[:10]

print("\n" + "="*120)
print("TOP 10 QUESTIONS INCORRECTES (Entropie la plus élevée - plus incertaines)")
print("="*120)

for i, r in enumerate(incorrect_sorted, 1):
    print(f"\n{i}. Question: {r['question']}")
    print(f"   Réponses vraies: {r['true_answers']}")
    print(f"   Consensus: '{r['consensus_answer']}' (freq: {r['consensus_frequency']:.1%})")
    print(f"   Accuracy: {r['accuracy']:.1%} | Entropie: {r['overall_mean_entropy']:.4f}")

## 11. GRAPHIQUE 1: Distribution de l'Entropie

In [ ]:
plt.figure(figsize=(12, 6))

plt.hist(all_mean_entropies, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
plt.axvline(global_mean_entropy, color='red', linestyle='--', linewidth=2,
            label=f'Moyenne: {global_mean_entropy:.4f} bits')

plt.xlabel('Entropie Moyenne par Question (bits)', fontsize=12)
plt.ylabel('Fréquence', fontsize=12)
plt.title('Distribution de l\'Entropie Token-Level', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Entropie moyenne: {global_mean_entropy:.4f} bits")
print(f"Écart-type: {global_std_entropy:.4f} bits")
print(f"Min: {min(all_mean_entropies):.4f} bits")
print(f"Max: {max(all_mean_entropies):.4f} bits")

## 12. GRAPHIQUE 2: Entropie selon la Justesse

In [ ]:
correct_entropies = [r['overall_mean_entropy'] for r in results if r['is_correct']]
incorrect_entropies = [r['overall_mean_entropy'] for r in results if not r['is_correct']]

plt.figure(figsize=(6, 4))

data_to_plot = [correct_entropies, incorrect_entropies]
labels = ['Correctes', 'Incorrectes']

bp = plt.boxplot(data_to_plot, labels=labels, patch_artist=True,
                 boxprops=dict(facecolor='lightblue', alpha=0.7),
                 medianprops=dict(color='red', linewidth=2))

plt.ylabel('Entropie Moyenne (bits)', fontsize=12)
plt.title('Distribution de l\'Entropie selon la Justesse des Réponses', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Entropie moyenne (Correctes): {np.mean(correct_entropies):.4f} bits")
print(f"Entropie moyenne (Incorrectes): {np.mean(incorrect_entropies):.4f} bits")
print(f"Différence: {np.mean(incorrect_entropies) - np.mean(correct_entropies):.4f} bits")

In [ ]:
plt.figure(figsize=(6, 4))

# Préparer les données pour violin plot
data_violin = []
categories = []

for ent in correct_entropies:
    data_violin.append(ent)
    categories.append('Correctes')

for ent in incorrect_entropies:
    data_violin.append(ent)
    categories.append('Incorrectes')

df_plot = pd.DataFrame({'Entropie': data_violin, 'Catégorie': categories})

# Violin plot avec seaborn
import seaborn as sns
sns.violinplot(data=df_plot, x='Catégorie', y='Entropie', palette=['lightgreen', 'lightcoral'])

# Ajouter les moyennes
means = [np.mean(correct_entropies), np.mean(incorrect_entropies)]
plt.plot([0, 1], means, 'ro-', linewidth=2, markersize=8, label='Moyenne')

plt.ylabel('Entropie Moyenne (bits)', fontsize=12)
plt.title('Distribution de l\'Entropie selon la Justesse des Réponses', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Entropie moyenne (Correctes): {np.mean(correct_entropies):.4f} bits")
print(f"Entropie moyenne (Incorrectes): {np.mean(incorrect_entropies):.4f} bits")
print(f"Différence: {np.mean(incorrect_entropies) - np.mean(correct_entropies):.4f} bits")

## 13. Analyse de Corrélation

In [ ]:
from scipy.stats import pearsonr, spearmanr

entropies_list = [r['overall_mean_entropy'] for r in results]
accuracies_list = [r['accuracy'] for r in results]

pearson_corr, pearson_p = pearsonr(entropies_list, accuracies_list)
spearman_corr, spearman_p = spearmanr(entropies_list, accuracies_list)

correct_mask = [r['is_correct'] for r in results]
correct_entropies_corr = [e for e, c in zip(entropies_list, correct_mask) if c]
incorrect_entropies_corr = [e for e, c in zip(entropies_list, correct_mask) if not c]

print("\n" + "="*80)
print("ANALYSE DE CORRÉLATION")
print("="*80)
print(f"Corrélation de Pearson (Entropie vs Accuracy): {pearson_corr:.4f} (p={pearson_p:.4f})")
print(f"Corrélation de Spearman (Entropie vs Accuracy): {spearman_corr:.4f} (p={spearman_p:.4f})")
print(f"\nEntropie moyenne (Réponses correctes): {np.mean(correct_entropies_corr):.4f} bits")
print(f"Entropie moyenne (Réponses incorrectes): {np.mean(incorrect_entropies_corr):.4f} bits")
print(f"Différence: {abs(np.mean(correct_entropies_corr) - np.mean(incorrect_entropies_corr)):.4f} bits")
print("="*80)

## 14. Scatter Plot Entropie vs Accuracy

In [ ]:
plt.figure(figsize=(10, 6))

correct_mask = [r['is_correct'] for r in results]
colors_scatter = ['green' if c else 'red' for c in correct_mask]

plt.scatter(entropies_list, accuracies_list, c=colors_scatter, alpha=0.6, s=50)

# Ligne de régression
z = np.polyfit(entropies_list, accuracies_list, 1)
p = np.poly1d(z)
plt.plot(sorted(entropies_list), p(sorted(entropies_list)), "b--", alpha=0.8, linewidth=2,
         label=f'Régression linéaire (r={pearson_corr:.3f})')

plt.xlabel('Entropie Moyenne (bits)', fontsize=12)
plt.ylabel('Accuracy (proportion)', fontsize=12)
plt.title('Corrélation Entropie vs Accuracy', fontsize=14, fontweight='bold')
plt.legend(['Régression', 'Correcte', 'Incorrecte'], fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15. 🆕 VISUALISATION EPR 1: Distribution des Densités (Correct vs Incorrect)

In [ ]:
# Adaptation du code 1: Distribution de l'EPR pour réponses correctes vs incorrectes
correct_epr = [r['overall_mean_entropy'] for r in results if r['is_correct']]
incorrect_epr = [r['overall_mean_entropy'] for r in results if not r['is_correct']]

mean_correct = np.mean(correct_epr)
mean_incorrect = np.mean(incorrect_epr)

plt.figure(figsize=(6, 4))
plt.hist(correct_epr, bins=40, density=True, alpha=0.6,
         label="Correct Answers")
plt.hist(incorrect_epr, bins=40, density=True, alpha=0.6,
         label="Incorrect Answers")

# Lignes verticales pour les moyennes
plt.axvline(mean_correct, linestyle="--", linewidth=2,
            label=f"Mean Correct: {mean_correct:.2f}")
plt.axvline(mean_incorrect, linestyle="--", linewidth=2,
            label=f"Mean Incorrect: {mean_incorrect:.2f}")

# Labels
plt.title("Distribution of EPR for WebQuestions (Qwen/Qwen2.5-3B-Instruct)", fontsize=18)
plt.xlabel("Entropy Production Rate (EPR)", fontsize=14)
plt.ylabel("Density", fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"\nMoyenne EPR (Correct): {mean_correct:.4f}")
print(f"Moyenne EPR (Incorrect): {mean_incorrect:.4f}")
print(f"Différence: {abs(mean_correct - mean_incorrect):.4f}")
print(f"Nombre de questions correctes: {len(correct_epr)}")
print(f"Nombre de questions incorrectes: {len(incorrect_epr)}")

## 16. 🆕 VISUALISATION EPR 2: Courbe ROC (EPR comme score de confiance)

In [ ]:
# Adaptation du code 2: ROC Curve avec EPR comme score de confiance
# Préparation des données au niveau des questions (consensus)
y_true_questions = np.array([1 if r['is_correct'] else 0 for r in results])
y_score_questions = -np.array([r['overall_mean_entropy'] for r in results])  # Négatif car EPR bas = confiance haute

# Calcul ROC au niveau des questions
roc_auc_questions = roc_auc_score(y_true_questions, y_score_questions)
fpr_questions, tpr_questions, _ = roc_curve(y_true_questions, y_score_questions)

# Visualisation
plt.figure(figsize=(6, 4))
plt.plot(fpr_questions, tpr_questions, label=f"ROC AUC = {roc_auc_questions:.3f}", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle="--", color='gray', label="Random")
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curve (EPR as confidence score)", fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nROC-AUC (niveau question): {roc_auc_questions:.4f}")
print(f"Interprétation: Une valeur > 0.5 indique que l'EPR est un indicateur utile de l'exactitude.")

# Statistiques supplémentaires
print(f"\nNombre de questions correctes: {y_true_questions.sum()}")
print(f"Nombre de questions incorrectes: {len(y_true_questions) - y_true_questions.sum()}")
print(f"Taux de base (accuracy): {y_true_questions.mean():.2%}")

## 17. 🆕 VISUALISATION EPR 3: Amélioration de l'Accuracy par Filtrage

In [ ]:
# Adaptation du code 3: Amélioration de l'accuracy en filtrant les réponses incertaines
# Préparation des données
df_filtrage = pd.DataFrame({
    'Entropie': [r['overall_mean_entropy'] for r in results],
    'numeric_correct': [1 if r['is_correct'] else 0 for r in results]
})

# On trie les réponses : de la plus "Sûre" (Entropie basse) à la plus "Incertaine" (Entropie haute)
df_sorted = df_filtrage.sort_values(by='Entropie', ascending=True).copy()

# On calcule l'accuracy cumulative
# Si on garde 100% des données, quelle est l'accuracy ? Si on garde 90% ? etc.
retained_ratios = np.linspace(0.1, 1.0, 100)  # De 10% à 100% de données gardées
accuracies_filtrage = []

for ratio in retained_ratios:
    cutoff_index = int(len(df_sorted) * ratio)
    if cutoff_index == 0:
        cutoff_index = 1  # Sécurité
    subset = df_sorted.iloc[:cutoff_index]
    acc = subset['numeric_correct'].mean()
    accuracies_filtrage.append(acc)

# Visualisation
plt.figure(figsize=(10, 6))
plt.plot(retained_ratios * 100, accuracies_filtrage, linewidth=3, color='#2c3e50',
         label='Performance après filtrage')

base_acc = df_filtrage['numeric_correct'].mean()
plt.axhline(y=base_acc, color='red', linestyle='--', linewidth=2,
            label=f'Accuracy actuelle ({base_acc:.1%})')

plt.title("Amélioration de l'Accuracy en filtrant les réponses incertaines", fontsize=14, fontweight='bold')
plt.xlabel("% de questions auxquelles on accepte de répondre (Taux de couverture)", fontsize=12)
plt.ylabel("Accuracy (Taux de bonnes réponses)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=11)
plt.xlim(100, 10)
plt.tight_layout()
plt.show()

# Analyse des gains potentiels
best_acc_idx = np.argmax(accuracies_filtrage)
best_coverage = retained_ratios[best_acc_idx] * 100
best_acc = accuracies_filtrage[best_acc_idx]

print(f"\nAccuracy de base (sans filtrage): {base_acc:.2%}")
print(f"Meilleure accuracy atteignable: {best_acc:.2%}")
print(f"Taux de couverture optimal: {best_coverage:.1f}%")
print(f"Gain en accuracy: +{(best_acc - base_acc)*100:.1f} points de pourcentage")

# Exemple pratique
coverage_50 = accuracies_filtrage[np.argmin(np.abs(retained_ratios - 0.5))]
coverage_70 = accuracies_filtrage[np.argmin(np.abs(retained_ratios - 0.7))]

print(f"\nExemples pratiques:")
print(f"- En répondant à 50% des questions (les plus sûres): {coverage_50:.2%} accuracy")
print(f"- En répondant à 70% des questions: {coverage_70:.2%} accuracy")

## 18. Export Complet des Résultats

In [ ]:
# Export détaillé
export_data = []
for i, r in enumerate(results, 1):
    export_data.append({
        'question_id': i,
        'question': r['question'],
        'true_answers': ' | '.join(r['true_answers']),
        'consensus_answer': r['consensus_answer'],
        'consensus_frequency': f"{r['consensus_frequency']:.2f}",
        'accuracy': f"{r['accuracy']:.2f}",
        'is_correct': r['is_correct'],
        'mean_entropy': f"{r['overall_mean_entropy']:.4f}",
        'unique_answers': len(set(r['generated_answers']))
    })

df_export = pd.DataFrame(export_data)
df_export.to_csv('webquestions_results.csv', index=False, encoding='utf-8')

print("Résultats exportés dans: webquestions_results.csv")
print(f"Nombre de lignes: {len(df_export)}")

## 19. RÉSUMÉ FINAL

In [ ]:
print("\n" + "="*80)
print("RÉSUMÉ FINAL")
print("="*80)

summary_final = {
    'Métrique': [
        'Dataset',
        'Modèle',
        'Température',
        'Échantillons/question',
        'Accuracy Globale',
        'Questions Réussies',
        'Entropie Moyenne',
        'Corrélation Entropie-Accuracy',
        'ROC-AUC (EPR)',
        'Accuracy Max (après filtrage)',
        'Taux de couverture optimal'
    ],
    'Valeur': [
        'WebQuestions',
        'Qwen/Qwen2.5-3B-Instruct',
        f'{TEMPERATURE}',
        f'{N_SAMPLES}',
        f'{global_accuracy:.2%}',
        f'{correct_questions}/{len(results)} ({question_accuracy:.2%})',
        f'{global_mean_entropy:.4f} ± {global_std_entropy:.4f} bits',
        f'{pearson_corr:.4f} (Pearson) / {spearman_corr:.4f} (Spearman)',
        f'{roc_auc_questions:.4f}',
        f'{best_acc:.2%}',
        f'{best_coverage:.1f}%'
    ]
}

df_summary_final = pd.DataFrame(summary_final)
print(df_summary_final.to_string(index=False))
print("="*80)

print("\n📊 INSIGHTS CLÉS:")
print(f"1. L'entropie moyenne est {'plus basse' if mean_correct < mean_incorrect else 'plus haute'} pour les réponses correctes")
print(f"2. L'AUC-ROC de {roc_auc_questions:.3f} indique que l'EPR est {'un bon' if roc_auc_questions > 0.6 else 'un' if roc_auc_questions > 0.5 else 'un faible'} indicateur de confiance")
print(f"3. En filtrant {100 - best_coverage:.1f}% des questions les plus incertaines, on peut améliorer l'accuracy de {base_acc:.1%} à {best_acc:.1%}")
print(f"4. La corrélation Pearson de {pearson_corr:.3f} suggère une relation {'forte' if abs(pearson_corr) > 0.7 else 'modérée' if abs(pearson_corr) > 0.4 else 'faible'} entre entropie et accuracy")